# Database Connection Test - Aurora & TiDB

This notebook tests connections to both database systems:
- **Aurora RDS MySQL**: Non-partitioned OLAP/Analytics database
- **TiDB**: Partitioned OLTP/Transactional database

In [1]:
# Import required libraries
import pymysql
import os
from dotenv import load_dotenv
from datetime import datetime

## Load Environment Variables

In [2]:
# Load environment variables from .env file
load_dotenv()

# Aurora RDS MySQL Configuration
AURORA_HOST = os.getenv('AURORA_HOST')
AURORA_PORT = int(os.getenv('AURORA_PORT', 3306))
AURORA_USER = os.getenv('AURORA_USER', 'admin')
AURORA_PASSWORD = os.getenv('AURORA_PASSWORD')
AURORA_DATABASE = os.getenv('AURORA_DATABASE', 'ai_state_management')

# TiDB Configuration
TIDB_HOST = os.getenv('TIDB_HOST', '127.0.0.1')
TIDB_PORT = int(os.getenv('TIDB_PORT', 3306))
TIDB_USER = os.getenv('TIDB_USER', 'root')
TIDB_PASSWORD = os.getenv('TIDB_PASSWORD', '')
TIDB_DATABASE = os.getenv('TIDB_DATABASE', 'ai_state_management')

print("Aurora Configuration:")
print(f"  Host: {AURORA_HOST}")
print(f"  Port: {AURORA_PORT}")
print(f"  User: {AURORA_USER}")
print(f"  Database: {AURORA_DATABASE}")

print("\nTiDB Configuration:")
print(f"  Host: {TIDB_HOST}")
print(f"  Port: {TIDB_PORT}")
print(f"  User: {TIDB_USER}")
print(f"  Database: {TIDB_DATABASE}")

print("\n✓ Environment variables loaded")

Aurora Configuration:
  Host: database-2.c6loqao2038b.us-east-1.rds.amazonaws.com
  Port: 3306
  User: admin
  Database: ai_state_management

TiDB Configuration:
  Host: 127.0.0.1
  Port: 3306
  User: root
  Database: ai_state_management

✓ Environment variables loaded


---

# Part 1: Aurora RDS MySQL Testing

## Connect to Aurora

In [3]:
# Connect to Aurora RDS MySQL
try:
    aurora_connection = pymysql.connect(
        host=AURORA_HOST,
        port=AURORA_PORT,
        user=AURORA_USER,
        password=AURORA_PASSWORD,
        database=AURORA_DATABASE,
        cursorclass=pymysql.cursors.DictCursor,
        connect_timeout=10
    )
    print(f"✓ Successfully connected to Aurora RDS MySQL")
    print(f"  Database: {AURORA_DATABASE}")
except Exception as e:
    print(f"✗ Connection failed: {e}")
    raise

✓ Successfully connected to Aurora RDS MySQL
  Database: ai_state_management


## Aurora Basic Queries

In [ ]:
# Get MySQL version
with aurora_connection.cursor() as cursor:
    cursor.execute("SELECT VERSION() as version")
    result = cursor.fetchone()
    print(f"MySQL Version: {result['version']}")

In [4]:
# Show current database
with aurora_connection.cursor() as cursor:
    cursor.execute("SELECT DATABASE() as current_db")
    result = cursor.fetchone()
    print(f"Current Database: {result['current_db']}")

Current Database: ai_state_management


In [5]:
# Show all databases
with aurora_connection.cursor() as cursor:
    cursor.execute("SHOW DATABASES")
    databases = cursor.fetchall()
    print("Available Databases:")
    for db in databases:
        print(f"  - {db['Database']}")

Available Databases:
  - ai_state_management
  - information_schema
  - mysql
  - performance_schema
  - sys


In [6]:
# Show tables in current database
with aurora_connection.cursor() as cursor:
    cursor.execute("SHOW TABLES")
    tables = cursor.fetchall()
    if tables:
        print("Tables in Aurora database:")
        for table in tables:
            table_name = list(table.values())[0]
            print(f"  - {table_name}")
    else:
        print("No tables found in current database")

Tables in Aurora database:
  - bots
  - memory_snapshots
  - messages
  - sessions
  - usage_stats
  - user_preferences
  - users


In [7]:
# Get connection information
with aurora_connection.cursor() as cursor:
    cursor.execute("SELECT USER() as user, CONNECTION_ID() as connection_id")
    result = cursor.fetchone()
    print(f"Connected as: {result['user']}")
    print(f"Connection ID: {result['connection_id']}")

Connected as: admin@98.43.32.82
Connection ID: 44


## Aurora Simple Query Test

In [8]:
# Test a simple SELECT query
with aurora_connection.cursor() as cursor:
    cursor.execute("SELECT 'Hello from Aurora!' as message, NOW() as timestamp")
    result = cursor.fetchone()
    print(f"Message: {result['message']}")
    print(f"Timestamp: {result['timestamp']}")

Message: Hello from Aurora!
Timestamp: 2026-04-03 01:57:08


## Aurora Data Loading Validation

Validate that test data was properly loaded into the Aurora database.

In [9]:
# Verify we're connected to the correct database
with aurora_connection.cursor() as cursor:
    cursor.execute("SELECT DATABASE() as current_db")
    result = cursor.fetchone()
    print(f"✓ Connected to database: {result['current_db']}")

✓ Connected to database: ai_state_management


In [10]:
# Check table counts
with aurora_connection.cursor() as cursor:
    tables = ['users', 'bots', 'sessions', 'messages', 'memory_snapshots']
    
    print("Aurora Table Row Counts:")
    print("=" * 50)
    
    aurora_table_counts = {}
    for table in tables:
        cursor.execute(f"SELECT COUNT(*) as count FROM {table}")
        result = cursor.fetchone()
        count = result['count']
        aurora_table_counts[table] = count
        status = "✓" if count > 0 else "✗"
        print(f"{status} {table:20s}: {count:,} rows")
    
    print("\n" + "=" * 50)
    print(f"Total records: {sum(aurora_table_counts.values()):,}")

Aurora Table Row Counts:
✓ users               : 100 rows
✓ bots                : 16 rows
✓ sessions            : 263 rows
✓ messages            : 7,811 rows
✓ memory_snapshots    : 1,000 rows

Total records: 9,190


In [11]:
# Validate referential integrity (foreign keys)
print("Aurora Referential Integrity:")
print("=" * 50)

with aurora_connection.cursor() as cursor:
    # Check sessions reference valid users
    cursor.execute("""
        SELECT COUNT(*) as orphaned 
        FROM sessions 
        WHERE user_id NOT IN (SELECT user_id FROM users)
    """)
    orphaned_sessions = cursor.fetchone()['orphaned']
    status = "✓" if orphaned_sessions == 0 else "✗"
    print(f"{status} Sessions with valid user_id: {orphaned_sessions} orphaned")
    
    # Check sessions reference valid bots
    cursor.execute("""
        SELECT COUNT(*) as orphaned 
        FROM sessions 
        WHERE bot_id NOT IN (SELECT bot_id FROM bots)
    """)
    orphaned_bots = cursor.fetchone()['orphaned']
    status = "✓" if orphaned_bots == 0 else "✗"
    print(f"{status} Sessions with valid bot_id: {orphaned_bots} orphaned")
    
    # Check messages reference valid sessions
    cursor.execute("""
        SELECT COUNT(*) as orphaned 
        FROM messages 
        WHERE session_id NOT IN (SELECT session_id FROM sessions)
    """)
    orphaned_messages = cursor.fetchone()['orphaned']
    status = "✓" if orphaned_messages == 0 else "✗"
    print(f"{status} Messages with valid session_id: {orphaned_messages} orphaned")
    
    # Check memory_snapshots reference valid sessions
    cursor.execute("""
        SELECT COUNT(*) as orphaned 
        FROM memory_snapshots 
        WHERE session_id NOT IN (SELECT session_id FROM sessions)
    """)
    orphaned_snapshots = cursor.fetchone()['orphaned']
    status = "✓" if orphaned_snapshots == 0 else "✗"
    print(f"{status} Memory snapshots with valid session_id: {orphaned_snapshots} orphaned")
    
    print("\n" + "=" * 50)
    total_orphaned = orphaned_sessions + orphaned_bots + orphaned_messages + orphaned_snapshots
    if total_orphaned == 0:
        print("✓ All foreign key relationships are valid!")
    else:
        print(f"✗ Found {total_orphaned} referential integrity issues")

Aurora Referential Integrity:
✓ Sessions with valid user_id: 0 orphaned
✓ Sessions with valid bot_id: 0 orphaned
✓ Messages with valid session_id: 0 orphaned
✓ Memory snapshots with valid session_id: 0 orphaned

✓ All foreign key relationships are valid!


In [ ]:
# Sample some data to verify content
print("Aurora Sample Data:")
print("=" * 50)

with aurora_connection.cursor() as cursor:
    # Sample users
    cursor.execute("SELECT * FROM users LIMIT 3")
    users = cursor.fetchall()
    print(f"\nUsers (showing 3 of {aurora_table_counts['users']}):")
    for user in users:
        print(f"  - {user['user_id']}: {user['username']} ({user['email']})")
    
    # Sample bots
    cursor.execute("SELECT * FROM bots LIMIT 3")
    bots = cursor.fetchall()
    print(f"\nBots (showing 3 of {aurora_table_counts['bots']}):")
    for bot in bots:
        print(f"  - {bot['bot_id']}: {bot['bot_name']} ({bot['bot_type']})")
    
    # Sample sessions
    cursor.execute("SELECT * FROM sessions LIMIT 3")
    sessions = cursor.fetchall()
    print(f"\nSessions (showing 3 of {aurora_table_counts['sessions']}):")
    for session in sessions:
        created = session.get('created_at') or session.get('created') or 'N/A'
        print(f"  - {session['session_id']}: user={session['user_id']}, bot={session['bot_id']}, created={created}")
    
    # Sample messages
    cursor.execute("SELECT * FROM messages LIMIT 3")
    messages = cursor.fetchall()
    print(f"\nMessages (showing 3 of {aurora_table_counts['messages']}):")
    for msg in messages:
        content_preview = msg['content'][:50] + "..." if len(msg['content']) > 50 else msg['content']
        print(f"  - {msg['message_id']}: role={msg['role']}, content=\"{content_preview}\"")
    
    # Check memory snapshots have embeddings
    cursor.execute("SELECT COUNT(*) as count FROM memory_snapshots WHERE embedding IS NOT NULL")
    embedded_count = cursor.fetchone()['count']
    print(f"\nMemory Snapshots with embeddings: {embedded_count:,} of {aurora_table_counts['memory_snapshots']:,}")
    
print("\n" + "=" * 50)
print("✓ Aurora data validation complete!")

## Close Aurora Connection

In [ ]:
# Close the Aurora connection
aurora_connection.close()
print("✓ Aurora connection closed")

---

# Part 2: TiDB Testing

## Connect to TiDB

In [13]:
# Connect to TiDB
try:
    tidb_connection = pymysql.connect(
        host=TIDB_HOST,
        port=TIDB_PORT,
        user=TIDB_USER,
        password=TIDB_PASSWORD,
        database=TIDB_DATABASE,
        cursorclass=pymysql.cursors.DictCursor,
        connect_timeout=10
    )
    print(f"✓ Successfully connected to TiDB")
    print(f"  Database: {TIDB_DATABASE}")
except Exception as e:
    print(f"✗ Connection failed: {e}")
    print("\nMake sure TiDB is running: docker-compose up -d")
    raise

✓ Successfully connected to TiDB
  Database: ai_state_management


## TiDB Basic Queries

In [14]:
# Get TiDB version
with tidb_connection.cursor() as cursor:
    cursor.execute("SELECT VERSION() as version")
    result = cursor.fetchone()
    print(f"TiDB Version: {result['version']}")

TiDB Version: 8.0.11-TiDB-v7.5.1


In [15]:
# Show current database
with tidb_connection.cursor() as cursor:
    cursor.execute("SELECT DATABASE() as current_db")
    result = cursor.fetchone()
    print(f"Current Database: {result['current_db']}")

Current Database: ai_state_management


In [16]:
# Show tables in TiDB database
with tidb_connection.cursor() as cursor:
    cursor.execute("SHOW TABLES")
    tables = cursor.fetchall()
    if tables:
        print("Tables in TiDB database:")
        for table in tables:
            table_name = list(table.values())[0]
            print(f"  - {table_name}")
    else:
        print("No tables found in current database")

Tables in TiDB database:
  - bots
  - memory_snapshots
  - messages
  - sessions
  - usage_stats
  - user_preferences
  - users


In [17]:
# Check TiDB partition information
with tidb_connection.cursor() as cursor:
    cursor.execute("""
        SELECT TABLE_NAME, PARTITION_NAME, PARTITION_EXPRESSION 
        FROM INFORMATION_SCHEMA.PARTITIONS 
        WHERE TABLE_SCHEMA = %s AND PARTITION_NAME IS NOT NULL
        ORDER BY TABLE_NAME, PARTITION_ORDINAL_POSITION
    """, (TIDB_DATABASE,))
    partitions = cursor.fetchall()
    if partitions:
        print("TiDB Partitioned Tables:")
        current_table = None
        for p in partitions:
            if p['TABLE_NAME'] != current_table:
                current_table = p['TABLE_NAME']
                print(f"\n  {current_table} (partitioned by {p['PARTITION_EXPRESSION']}):")
            print(f"    - {p['PARTITION_NAME']}")
    else:
        print("No partitioned tables found")

No partitioned tables found


## TiDB Data Loading Validation

Validate that test data was properly loaded into the TiDB database.

In [18]:
# Check table counts
with tidb_connection.cursor() as cursor:
    tables = ['users', 'bots', 'sessions', 'messages', 'memory_snapshots']
    
    print("TiDB Table Row Counts:")
    print("=" * 50)
    
    tidb_table_counts = {}
    for table in tables:
        cursor.execute(f"SELECT COUNT(*) as count FROM {table}")
        result = cursor.fetchone()
        count = result['count']
        tidb_table_counts[table] = count
        status = "✓" if count > 0 else "✗"
        print(f"{status} {table:20s}: {count:,} rows")
    
    print("\n" + "=" * 50)
    print(f"Total records: {sum(tidb_table_counts.values()):,}")

TiDB Table Row Counts:
✓ users               : 103 rows
✓ bots                : 16 rows
✓ sessions            : 263 rows
✓ messages            : 7,811 rows
✓ memory_snapshots    : 1,000 rows

Total records: 9,193


In [19]:
# Validate referential integrity (foreign keys)
print("TiDB Referential Integrity:")
print("=" * 50)

with tidb_connection.cursor() as cursor:
    # Check sessions reference valid users
    cursor.execute("""
        SELECT COUNT(*) as orphaned 
        FROM sessions 
        WHERE user_id NOT IN (SELECT user_id FROM users)
    """)
    orphaned_sessions = cursor.fetchone()['orphaned']
    status = "✓" if orphaned_sessions == 0 else "✗"
    print(f"{status} Sessions with valid user_id: {orphaned_sessions} orphaned")
    
    # Check sessions reference valid bots
    cursor.execute("""
        SELECT COUNT(*) as orphaned 
        FROM sessions 
        WHERE bot_id NOT IN (SELECT bot_id FROM bots)
    """)
    orphaned_bots = cursor.fetchone()['orphaned']
    status = "✓" if orphaned_bots == 0 else "✗"
    print(f"{status} Sessions with valid bot_id: {orphaned_bots} orphaned")
    
    # Check messages reference valid sessions
    cursor.execute("""
        SELECT COUNT(*) as orphaned 
        FROM messages 
        WHERE session_id NOT IN (SELECT session_id FROM sessions)
    """)
    orphaned_messages = cursor.fetchone()['orphaned']
    status = "✓" if orphaned_messages == 0 else "✗"
    print(f"{status} Messages with valid session_id: {orphaned_messages} orphaned")
    
    # Check memory_snapshots reference valid sessions
    cursor.execute("""
        SELECT COUNT(*) as orphaned 
        FROM memory_snapshots 
        WHERE session_id NOT IN (SELECT session_id FROM sessions)
    """)
    orphaned_snapshots = cursor.fetchone()['orphaned']
    status = "✓" if orphaned_snapshots == 0 else "✗"
    print(f"{status} Memory snapshots with valid session_id: {orphaned_snapshots} orphaned")
    
    print("\n" + "=" * 50)
    total_orphaned = orphaned_sessions + orphaned_bots + orphaned_messages + orphaned_snapshots
    if total_orphaned == 0:
        print("✓ All foreign key relationships are valid!")
    else:
        print(f"✗ Found {total_orphaned} referential integrity issues")

TiDB Referential Integrity:
✓ Sessions with valid user_id: 0 orphaned
✓ Sessions with valid bot_id: 0 orphaned
✓ Messages with valid session_id: 0 orphaned
✓ Memory snapshots with valid session_id: 0 orphaned

✓ All foreign key relationships are valid!


In [20]:
# Sample some data to verify content
print("TiDB Sample Data:")
print("=" * 50)

with tidb_connection.cursor() as cursor:
    # Sample users
    cursor.execute("SELECT * FROM users LIMIT 3")
    users = cursor.fetchall()
    print(f"\nUsers (showing 3 of {tidb_table_counts['users']}):")
    for user in users:
        print(f"  - {user['user_id']}: {user['username']} ({user['email']})")
    
    # Sample bots
    cursor.execute("SELECT * FROM bots LIMIT 3")
    bots = cursor.fetchall()
    print(f"\nBots (showing 3 of {tidb_table_counts['bots']}):")
    for bot in bots:
        print(f"  - {bot['bot_id']}: {bot['bot_name']} ({bot['bot_type']})")
    
    # Sample sessions
    cursor.execute("SELECT * FROM sessions LIMIT 3")
    sessions = cursor.fetchall()
    print(f"\nSessions (showing 3 of {tidb_table_counts['sessions']}):")
    for session in sessions:
        created = session.get('created_at') or session.get('created') or 'N/A'
        print(f"  - {session['session_id']}: user={session['user_id']}, bot={session['bot_id']}, created={created}")
    
    # Sample messages
    cursor.execute("SELECT * FROM messages LIMIT 3")
    messages = cursor.fetchall()
    print(f"\nMessages (showing 3 of {tidb_table_counts['messages']}):")
    for msg in messages:
        content_preview = msg['content'][:50] + "..." if len(msg['content']) > 50 else msg['content']
        print(f"  - {msg['message_id']}: role={msg['role']}, content=\"{content_preview}\"")
    
    # Check memory snapshots have embeddings
    cursor.execute("SELECT COUNT(*) as count FROM memory_snapshots WHERE embedding IS NOT NULL")
    embedded_count = cursor.fetchone()['count']
    print(f"\nMemory Snapshots with embeddings: {embedded_count:,} of {tidb_table_counts['memory_snapshots']:,}")
    
print("\n" + "=" * 50)
print("✓ TiDB data validation complete!")

TiDB Sample Data:

Users (showing 3 of 103):
  - 01a9b14d-f43d-4b29-a03a-f7ebc03d3434: yara.hernandez847 (yara.hernandez847@example.com)
  - 04339916-7719-4f76-b663-20a1ffde3198: henry.davis877 (henry.davis877@example.com)
  - 049f1a7e-ea67-4fd4-b68b-86bea765086b: diana.rodriguez779 (diana.rodriguez779@example.com)

Bots (showing 3 of 16):
  - assistant-bot-1: General Assistant (assistant)
  - career-bot-12: Career Counselor (career)
  - coding-bot-9: Code Helper (coding)

Sessions (showing 3 of 263):
  - 000339a6-6650-43c8-bf18-f3a677b45c5c: user=6e55c535-75e7-4d28-a6a1-029a3eb56ef8, bot=support-bot-2, created=N/A
  - 00b3632c-83ea-4168-92a4-b5f76c95dd9d: user=9e206ff3-a454-4db1-b5d6-d7f16c41cb1b, bot=assistant-bot-1, created=N/A
  - 00da6827-f719-4dfd-a60e-4c00c8ad1d41: user=e13760b2-f251-481c-b095-cb9d2d651a4e, bot=finance-bot-7, created=N/A

Messages (showing 3 of 7811):
  - 1: role=user, content="How do I reset my password?"
  - 2: role=assistant, content="Here's what I recommend:

## Close TiDB Connection

In [21]:
# Close the TiDB connection
tidb_connection.close()
print("✓ TiDB connection closed")

✓ TiDB connection closed


---

# Summary

Comparison of both database systems.

In [22]:
# Compare data between Aurora and TiDB
print("Database Comparison")
print("=" * 70)
print(f"{'Table':<25} {'Aurora':<20} {'TiDB':<20} {'Match':<5}")
print("=" * 70)

tables = ['users', 'bots', 'sessions', 'messages', 'memory_snapshots']
all_match = True

for table in tables:
    aurora_count = aurora_table_counts.get(table, 0)
    tidb_count = tidb_table_counts.get(table, 0)
    match = "✓" if aurora_count == tidb_count else "✗"
    if aurora_count != tidb_count:
        all_match = False
    print(f"{table:<25} {aurora_count:<20,} {tidb_count:<20,} {match:<5}")

print("=" * 70)
aurora_total = sum(aurora_table_counts.values())
tidb_total = sum(tidb_table_counts.values())
print(f"{'TOTAL':<25} {aurora_total:<20,} {tidb_total:<20,} {'✓' if all_match else '✗':<5}")

if all_match:
    print("\n✓ Both databases have identical record counts!")
else:
    print("\n✗ Databases have different record counts")

Database Comparison
Table                     Aurora               TiDB                 Match
users                     100                  103                  ✗    
bots                      16                   16                   ✓    
sessions                  263                  263                  ✓    
messages                  7,811                7,811                ✓    
memory_snapshots          1,000                1,000                ✓    
TOTAL                     9,190                9,193                ✗    

✗ Databases have different record counts


---

# CDC Replication Test

Test that changes in Aurora are replicated to TiDB via CDC (Change Data Capture).

In [ ]:
import time
from datetime import datetime
import uuid

# Generate a unique test username
test_user_id = str(uuid.uuid4())
test_username = f"cdc.test.{int(time.time())}"
test_email = "cdctest@example.com"

print("CDC Replication Test")
print("=" * 70)
print(f"Test username: {test_username}")
print(f"Test user_id: {test_user_id}")

# Insert test user into Aurora (reuses existing connection)
print("\n1. Inserting test user into Aurora...")
try:
    # Reconnect Aurora if needed
    aurora_connection = pymysql.connect(
        host=AURORA_HOST,
        port=AURORA_PORT,
        user=AURORA_USER,
        password=AURORA_PASSWORD,
        database=AURORA_DATABASE,
        cursorclass=pymysql.cursors.DictCursor
    )
    
    with aurora_connection.cursor() as cursor:
        sql = """INSERT INTO users (user_id, username, email, created_at) 
                 VALUES (%s, %s, %s, NOW())"""
        cursor.execute(sql, (test_user_id, test_username, test_email))
        aurora_connection.commit()
        print(f"   ✓ User inserted into Aurora")
        
except Exception as e:
    print(f"   ✗ Error inserting into Aurora: {e}")

CDC Replication Test
Test username: cdc.test.1775182509
Test user_id: 7f6c6797-1c74-4ce6-bc58-97dae5f679d7

1. Inserting test user into Aurora...
   ✓ User inserted into Aurora


In [24]:
# Wait for CDC replication
print("\n2. Waiting for CDC replication...")
wait_seconds = 5
for i in range(wait_seconds, 0, -1):
    print(f"   {i} seconds remaining...", end="\r")
    time.sleep(1)
print("   ✓ Wait complete" + " " * 30)


2. Waiting for CDC replication...
   ✓ Wait complete                              


In [ ]:
# Check if user replicated to TiDB (reuses existing connection)
print("\n3. Checking TiDB for replicated user...")
try:
    # Reconnect TiDB if needed
    tidb_connection = pymysql.connect(
        host=TIDB_HOST,
        port=TIDB_PORT,
        user=TIDB_USER,
        password=TIDB_PASSWORD,
        database=TIDB_DATABASE,
        cursorclass=pymysql.cursors.DictCursor
    )
    
    with tidb_connection.cursor() as cursor:
        sql = """SELECT user_id, username, email, created_at 
                 FROM users 
                 WHERE username = %s"""
        cursor.execute(sql, (test_username,))
        result = cursor.fetchone()
        
        if result:
            print(f"   ✓ User found in TiDB!")
            print(f"   - user_id: {result['user_id']}")
            print(f"   - username: {result['username']}")
            print(f"   - email: {result['email']}")
            print(f"   - created_at: {result['created_at']}")
            print(f"\n✓ CDC replication is working correctly!")
        else:
            print(f"   ✗ User NOT found in TiDB")
            print(f"\n✗ CDC replication may not be working")
            
except Exception as e:
    print(f"   ✗ Error checking TiDB: {e}")

print("=" * 70)


3. Checking TiDB for replicated user...
   ✗ User NOT found in TiDB

✗ CDC replication may not be working


### Cleanup Test Data

Optionally remove the test user from both databases.

In [ ]:
# Uncomment to clean up test data (reuses existing connections)
# print("Cleaning up test user...")
# 
# # Delete from Aurora
# with aurora_connection.cursor() as cursor:
#     cursor.execute("DELETE FROM users WHERE username = %s", (test_username,))
#     aurora_connection.commit()
# print("✓ Test user removed from Aurora")
# 
# # Wait for CDC to replicate deletion
# time.sleep(5)
# 
# # Verify deletion in TiDB
# with tidb_connection.cursor() as cursor:
#     cursor.execute("SELECT COUNT(*) as count FROM users WHERE username = %s", (test_username,))
#     if cursor.fetchone()['count'] == 0:
#         print("✓ Test user removed from TiDB via CDC")

#     else:#         print("✗ Test user still exists in TiDB")